# Lakehouse: Gold と Iceberg を対話操作する

この notebook は operator 用です。接続値は private env から読みます。秘密値を表示する cell はありません。Spark の書込先は `notebook_demo` だけです。

In [ ]:
import pandas as pd
from lakehouse_session import read_gold, spark_session

columns, rows = read_gold("factory_daily_production", limit=20)
pd.DataFrame(rows, columns=columns)

In [ ]:
columns, rows = read_gold("sales_fulfillment", limit=20)
pd.DataFrame(rows, columns=columns)

## Spark / Iceberg の小さな読書き

Spark 4.1.4 と Iceberg 1.12.0 を使います。compute 有効時は常駐 Worker、無効時は local[2] で実行します。通常の Bronze/Silver/Gold は更新しません。

In [ ]:
spark = spark_session()
print("Spark", spark.version)
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.notebook_demo")
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.notebook_demo.notes (
        id BIGINT, label STRING
    ) USING iceberg
    TBLPROPERTIES ('format-version' = '2')
""")
spark.sql("""
    MERGE INTO lakehouse.notebook_demo.notes AS target
    USING (SELECT 1L AS id, 'operator notebook demo' AS label) AS source
    ON target.id = source.id
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM lakehouse.notebook_demo.notes ORDER BY id").show()

In [ ]:
spark.stop()
# Kernel を終了すると、残った Python state も破棄されます。
# Notebook の保存は外部 workspace に残ります。